### Random forest 

In [1]:
import seaborn as sns

df = sns.load_dataset("iris")
df

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [2]:
df['species'].unique()

array(['setosa', 'versicolor', 'virginica'], dtype=object)

In [3]:
import pandas as pd 
df['species'] = df['species'].map({
    'setosa': 0,
    'versicolor': 1,
    'virginica': 2
})
df.sample(5) # sample is used to get a random sample of the data

,sepal_length,sepal_width,petal_length,petal_width,species
5,5.4,3.9,1.7,0.4,0
116,6.5,3.0,5.5,1.8,2
125,7.2,3.2,6.0,1.8,2
84,5.4,3.0,4.5,1.5,1
68,6.2,2.2,4.5,1.5,1


In [4]:
X = df.drop(columns=['species'], axis=1)
y = df['species']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)
# stratify=y ensures that the split is stratified, meaning that the proportion of each class in the target variable is maintained in both the training and test sets.
# random_state=42 ensures that the split is reproducible.
y_train.value_counts()

species
0    45
2    45
1    45
Name: count, dtype: int64

In [5]:
y_train.unique()

array([0, 2, 1])

In [6]:
from sklearn.tree import DecisionTreeClassifier

dt_clf = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
dt_clf.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=3, random_state=42)

In [7]:
from sklearn.metrics import accuracy_score

y_pred = dt_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.9333333333333333

### Hyperparameter tuning

In [8]:
# using gridsearchcv
from sklearn.model_selection import GridSearchCV

params = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [1, 2, 3],
    'random_state': [42]
}

grid = GridSearchCV(DecisionTreeClassifier(), param_grid=params)
grid.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [1, 2, 3], 'random_state': [42]})

In [9]:
grid.best_params_

{'criterion': 'gini', 'max_depth': 3, 'random_state': 42}

In [10]:
grid.best_score_

np.float64(0.9703703703703704)

In [11]:
grid.best_estimator_

DecisionTreeClassifier(max_depth=3, random_state=42)

In [12]:
grid.best_estimator_.predict(X_test)

array([1, 2, 2, 1, 2, 0, 0, 0, 2, 1, 0, 2, 1, 2, 0])

## Randomized Search

In [14]:
from sklearn.model_selection import RandomizedSearchCV
params = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': range(1, 50),
    'random_state': [42]
}

rnd_search = RandomizedSearchCV(DecisionTreeClassifier(), param_distributions=params)
rnd_search.fit(X_train, y_train)


RandomizedSearchCV(estimator=DecisionTreeClassifier(),
                   param_distributions={'criterion': ['gini', 'entropy',
                                                      'log_loss'],
                                        'max_depth': range(1, 50),
                                        'random_state': [42]})

In [15]:
rnd_search.best_params_

{'random_state': 42, 'max_depth': 27, 'criterion': 'gini'}

In [16]:
rnd_search.best_score_

np.float64(0.9555555555555555)

In [18]:
rnd_search.best_estimator_.predict(X_test)

array([1, 1, 2, 1, 2, 0, 0, 0, 2, 1, 0, 2, 1, 2, 0])

In [19]:
y_pred = rnd_search.best_estimator_.predict(X_test)
accuracy_score(y_test, y_pred)

0.8666666666666667

## Cross-Validation

In [21]:
from sklearn.model_selection import cross_val_score

acc = cross_val_score(DecisionTreeClassifier(), X_train, y_train, scoring='accuracy', cv=5 )

In [22]:
acc 

array([0.96296296, 1.        , 0.96296296, 0.92592593, 0.92592593])